# Imports and Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("seeds_data.csv")
X = df.iloc[:, :-1].values.astype(float)
y = df.iloc[:, -1].values

mean = X.mean(axis=0)
std = X.std(axis=0)
std[std == 0] = 1
X_scaled = (X - mean) / std

print(df.head())
print("Dataset shape:", df.shape)

# K-Means from Scratch

In [ ]:
def kmeans(X, k, max_iter=100):
    np.random.seed(42)
    centers = X[np.random.choice(len(X), k, replace=False)].copy()

    for _ in range(max_iter):
        distances = np.sqrt(((X[:, None] - centers) ** 2).sum(axis=2))
        labels = np.argmin(distances, axis=1)
        new_centers = np.array([
            X[labels == i].mean(axis=0) if np.any(labels == i) else centers[i]
            for i in range(k)
        ])
        if np.allclose(centers, new_centers):
            centers = new_centers
            break
        centers = new_centers

    distances = np.sqrt(((X[:, None] - centers) ** 2).sum(axis=2))
    labels = np.argmin(distances, axis=1)
    wcss = sum(((X[labels == i] - centers[i]) ** 2).sum() for i in range(k))
    return labels, centers, wcss

# Elbow Method

In [ ]:
wcss = []
for k in range(1, 11):
    labels, centers, cost = kmeans(X_scaled, k)
    wcss.append(cost)

plt.plot(range(1, 11), wcss, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.grid()
plt.show()

# Fit K-Means and Print Centers

In [ ]:
k = 3
kmeans_labels, centers, cost = kmeans(X_scaled, k)
print("Cluster centers (scaled data):")
print(centers)
print("WCSS:", cost)
print("Cluster counts:", np.bincount(kmeans_labels))

# DBSCAN Parameter Graph

In [ ]:
min_samples = 5
distances = np.sqrt(((X_scaled[:, None] - X_scaled[None, :]) ** 2).sum(axis=2))
k_distances = np.sort(np.sort(distances, axis=1)[:, min_samples - 1])

plt.plot(k_distances)
plt.xlabel("Points Sorted by Distance")
plt.ylabel("Distance to 5th Neighbor")
plt.title("DBSCAN K-Distance Graph")
plt.grid()
plt.show()

# DBSCAN from Scratch

In [ ]:
def dbscan(X, eps, min_samples):
    n = len(X)
    labels = np.full(n, -1)
    visited = np.zeros(n, dtype=bool)
    cluster = 0
    distances = np.sqrt(((X[:, None] - X[None, :]) ** 2).sum(axis=2))

    for i in range(n):
        if visited[i]:
            continue
        visited[i] = True
        neighbors = np.where(distances[i] <= eps)[0]
        if len(neighbors) < min_samples:
            continue

        labels[i] = cluster
        queue = list(neighbors)
        queued = set(queue)
        j = 0

        while j < len(queue):
            point = queue[j]
            j += 1
            if not visited[point]:
                visited[point] = True
                new_neighbors = np.where(distances[point] <= eps)[0]
                if len(new_neighbors) >= min_samples:
                    for p in new_neighbors:
                        if p not in queued:
                            queue.append(p)
                            queued.add(p)
            if labels[point] == -1:
                labels[point] = cluster
        cluster += 1

    return labels

# Fit DBSCAN

In [ ]:
eps = 1.2
dbscan_labels = dbscan(X_scaled, eps, min_samples)
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = np.sum(dbscan_labels == -1)

print("Number of Clusters:", n_clusters)
print("Number of Noise Points:", n_noise)
print("Cluster labels:", np.unique(dbscan_labels))

# PCA from Scratch

In [ ]:
def pca(X, n_components=2):
    X_centered = X - X.mean(axis=0)
    cov = np.cov(X_centered, rowvar=False)
    values, vectors = np.linalg.eigh(cov)
    order = np.argsort(values)[::-1]
    values = values[order]
    vectors = vectors[:, order[:n_components]]
    X_pca = X_centered @ vectors
    explained_variance = values[:n_components] / values.sum()
    return X_pca, explained_variance

X_pca, explained_variance = pca(X_scaled)
print("Explained variance ratio:", explained_variance)
print("Total explained variance:", explained_variance.sum())

# Visualize Both Algorithms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap="viridis")
axes[0].set_title("K-Means Clustering")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=dbscan_labels, cmap="viridis")
noise = dbscan_labels == -1
axes[1].scatter(X_pca[noise, 0], X_pca[noise, 1], color="red", marker="x", s=100, label="Noise")
axes[1].set_title("DBSCAN Clustering")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].legend()

plt.tight_layout()
plt.show()

# Evaluation Without Scikit-learn

In [ ]:
def ari_score(y_true, y_pred):
    table = pd.crosstab(y_true, y_pred).values
    choose2 = lambda n: n * (n - 1) / 2
    sum_cells = sum(choose2(n) for n in table.flatten())
    sum_rows = sum(choose2(n) for n in table.sum(axis=1))
    sum_cols = sum(choose2(n) for n in table.sum(axis=0))
    total = choose2(table.sum())
    if total == 0:
        return 1.0
    expected = sum_rows * sum_cols / total
    maximum = (sum_rows + sum_cols) / 2
    if maximum == expected:
        return 1.0
    return (sum_cells - expected) / (maximum - expected)

print("K-Means ARI:", ari_score(y, kmeans_labels))
print("DBSCAN ARI:", ari_score(y, dbscan_labels))

print("\nK-Means cross-tabulation:")
print(pd.crosstab(y, kmeans_labels))

print("\nDBSCAN cross-tabulation:")
print(pd.crosstab(y, dbscan_labels))